In [ ]:
import os

base_path = "/content/drive/MyDrive/spectrograms"  # exact path

folders = ["HC", "PD"]

for folder in folders:
    folder_path = os.path.join(base_path, folder)

    # check folder exists
    if not os.path.exists(folder_path):
        print(f"❌ Folder not found: {folder_path}")
        continue

    # count image files
    count = sum(
        1 for f in os.listdir(folder_path)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    )

    print(f"📂 {folder} folder image count: {count}")


📂 HC folder image count: 1000
📂 PD folder image count: 1000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
from PIL import Image
import random
from torchvision import transforms

# ============================
# PATHS
# ============================
base_path = "/content/drive/MyDrive/spectrograms"

folders = ["HC", "PD"]

# ============================
# AUGMENTATION PIPELINE
# ============================
augment = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0))
])

# ============================
# FUNCTION TO CREATE 1000 IMAGES
# ============================
def make_1000_images(folder_path):
    images = [f for f in os.listdir(folder_path) if f.endswith((".png",".jpg",".jpeg"))]
    current_count = len(images)

    print(f"Found: {current_count} images in {folder_path}")

    target_count = 1000
    needed = target_count - current_count

    if needed <= 0:
        print("Already 1000 or more images. Skipping.")
        return

    print(f"Generating {needed} augmented images...")

    for i in range(needed):
        # pick a random original image
        img_name = random.choice(images)
        img_path = os.path.join(folder_path, img_name)

        img = Image.open(img_path).convert("RGB")
        aug_img = augment(img)

        save_path = os.path.join(folder_path, f"aug_{i}.png")
        aug_img.save(save_path)

    print(f"Completed: Now {target_count} images in {folder_path}")

# ============================
# PROCESS BOTH FOLDERS
# ============================
for folder in folders:
    make_1000_images(os.path.join(base_path, folder))


Found: 298 images in /content/drive/MyDrive/spectrograms/HC
Generating 702 augmented images...
Completed: Now 1000 images in /content/drive/MyDrive/spectrograms/HC
Found: 298 images in /content/drive/MyDrive/spectrograms/PD
Generating 702 augmented images...
Completed: Now 1000 images in /content/drive/MyDrive/spectrograms/PD


In [ ]:
# ============================================
# 📌 Deep Learning Spectrogram Processing Pipeline
# ============================================

import os
from PIL import Image, ImageFilter, ImageOps
import numpy as np
import torchvision.transforms as transforms

# ============================================
# PATHS
# ============================================
input_base = "/content/drive/MyDrive/spectrograms"
output_base = "/content/drive/MyDrive/processed_data"
folders = ["HC", "PD"]

# Create processed_data directory structure
for folder in folders:
    os.makedirs(os.path.join(output_base, folder), exist_ok=True)

# ============================================
# TRANSFORMS
# ============================================
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),       # CNN-friendly
    transforms.ToTensor(),               # convert to tensor
    transforms.Normalize(mean=[0.5], std=[0.5])  # standard normalization
])

# ============================================
# IMAGE ENHANCEMENT FUNCTION
# ============================================
def enhance_spectrogram(img):
    """Preprocess each spectrogram image"""

    # 1) Convert to grayscale
    img = img.convert("L")

    # 2) Noise reduction (median filter)
    img = img.filter(ImageFilter.MedianFilter(size=3))

    # 3) Edge detection
    edges = img.filter(ImageFilter.FIND_EDGES)

    # Combine original + edges for better contrast
    img = Image.blend(img, edges, alpha=0.3)

    # 4) Convert to numpy
    img_np = np.array(img).astype("float32")

    # 5) Log amplitude compression
    img_np = np.log1p(img_np)

    # 6) Min-max scaling to 0-1
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

    # 7) Contrast enhancement
    img_np = np.clip(1.5 * (img_np - 0.5) + 0.5, 0, 1)

    # Convert back to PIL
    img = Image.fromarray((img_np * 255).astype("uint8"))

    return img

# ============================================
# PROCESSING LOOP
# ============================================
for folder in folders:
    input_path = os.path.join(input_base, folder)
    output_path = os.path.join(output_base, folder)

    images = [f for f in os.listdir(input_path) if f.lower().endswith((".png", ".jpg", ".jpeg"))]

    print(f"Processing {folder}: {len(images)} images")

    for img_name in images:
        img_path = os.path.join(input_path, img_name)
        img = Image.open(img_path)

        # Enhancement
        img = enhance_spectrogram(img)

        # Deep learning preprocess
        img_tensor = preprocess(img)

        # Save back as image
        save_img = transforms.ToPILImage()(img_tensor)
        save_img.save(os.path.join(output_path, img_name))

    print(f"✅ Completed: {folder} → saved in processed_data/{folder}")


Processing HC: 1000 images
✅ Completed: HC → saved in processed_data/HC
Processing PD: 1000 images
✅ Completed: PD → saved in processed_data/PD


In [ ]:
import os

# processed_data path
processed_base = "/content/drive/MyDrive/processed_data"
folders = ["HC", "PD"]

for folder in folders:
    folder_path = os.path.join(processed_base, folder)
    images = [f for f in os.listdir(folder_path) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    print(f"{folder}: {len(images)} images")


HC: 1000 images
PD: 1000 images


In [ ]:
import os, shutil, random

data_dir = "/content/drive/MyDrive/processed_data"
classes = ["HC", "PD"]
split_ratio = 0.8  # 80% train, 20% test

for c in classes:
    class_dir = os.path.join(data_dir, c)
    images = os.listdir(class_dir)
    random.shuffle(images)
    split = int(len(images)*split_ratio)

    train_dir = os.path.join(data_dir,"train",c)
    test_dir  = os.path.join(data_dir,"test",c)
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    for img in images[:split]:
        shutil.copy(os.path.join(class_dir,img), train_dir)
    for img in images[split:]:
        shutil.copy(os.path.join(class_dir,img), test_dir)

print("✔ Dataset successfully split!")
